In [15]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import natural_units as nu
from scipy.special import erf
from scipy.integrate import quad
from scipy.interpolate import RegularGridInterpolator
# multi-core/thread:
import concurrent.futures

In [16]:
# HgCdTe form factors' parameters
N_q   = 800
N_E   = 300
dE    = 0.05 * nu.eV
dq    = 0.01 * nu.aEM * nu.mElectron
E_max = N_E * dE
q_max = N_q * dq
energy_gap = 0.234 * nu.eV
epsilon    = 3 * energy_gap
M_cell     = 301.74 * nu.AMU
Q_max      = np.floor((E_max - energy_gap + epsilon) / epsilon)
# Read form factor files
ff_hgte_grid = np.zeros((N_q,N_E))
ff_cdte_grid = np.zeros((N_q,N_E))
hgte_data_file = np.loadtxt('../data/form_factors/C.HgTe137.dat')
cdte_data_file = np.loadtxt('../data/form_factors/C.CdTe137.dat')
lin2_data_file = np.loadtxt('../data/form_factors/Lin2_HgCdTe.txt')
i=0
for Ei in range(N_E):
    for qi in range(N_q):
        ff_hgte_grid[qi, Ei] = hgte_data_file[i]
        ff_cdte_grid[qi, Ei] = cdte_data_file[i]
        i += 1

# correct by Lindhard:
ff_hgte_grid = ff_hgte_grid / lin2_data_file
ff_cdte_grid = ff_cdte_grid / lin2_data_file

# Interplot form factors
q_grid = np.linspace(dq, q_max, N_q)
E_grid = np.linspace(dE, E_max, N_E)
ff_hgte = RegularGridInterpolator((q_grid, E_grid), ff_hgte_grid)
ff_cdte = RegularGridInterpolator((q_grid, E_grid), ff_cdte_grid)

In [17]:
# Halo DM parameters, just in case
rho_DM  = 0.3 * nu.GeV / nu.cm**3
# frac_DM = 1.0   # Adjust later, fix it to be 1 for now.
vesc    = 544.0 * nu.km / nu.sec
v0      = 238.0 * nu.km / nu.sec
v_Earth = 250.2 * nu.km / nu.sec

# DM speed distribution
Nesc = np.pi * v0 * v0 * (np.sqrt(np.pi) * v0 * erf(vesc / v0) - 2 * vesc * np.exp(-vesc * vesc / v0 / v0))
def f_halo(v, vEarth = v_Earth):
	return np.pi * v * v0 * v0 / Nesc / vEarth * (2 * np.exp(-(v * v + vEarth * vEarth) / v0 / v0) * np.sinh(2 * v * vEarth / v0 / v0) + (np.exp(-np.power(v + vEarth, 2.0) / v0 / v0) - np.exp(-vesc * vesc / v0 / v0)) * np.heaviside(np.abs(v + vEarth) - vesc, 0) - (np.exp(-np.power(v - vEarth, 2.0) / v0 / v0) - np.exp(-vesc * vesc / v0 / v0)) * np.heaviside(np.abs(v - vEarth) - vesc, 0))

def EtaFunction(vMin, vE = v_Earth):
    xMin = vMin / v0
    xEsc = vesc / v0
    xE = vE / v0
    eta = 0.0
    if xMin > xE + xEsc:
        eta = 0.0
    elif np.abs(xMin - xE - xEsc) < 1e-8:
        eta = 0.0
    elif xMin > np.abs(xE - xEsc):
        eta = np.power(np.pi, 1.5) * v0 * v0 / 2.0 / Nesc / xE * (erf(xEsc) - erf(xMin - xE) - 2.0 / np.sqrt(np.pi) * (xE + xEsc - xMin) * np.exp(-xEsc * xEsc))
    elif xEsc > xE:
        eta = np.power(np.pi, 1.5) * v0 * v0 / 2.0 / Nesc / xE * (erf(xMin + xE) - erf(xMin - xE) - 4.0 / np.sqrt(np.pi) * xE * np.exp(-xEsc * xEsc))
    else:
        eta = 1.0 / v0 / xE
    return eta

In [18]:
# To determine light or heavy mediator, just in case
mA = 0.0

def F_DM(q):
    return ((nu.aEM*nu.mElectron)**2 + mA**2)/(q**2 + mA**2)

def v_min(q, Ee, mDM):
    return Ee/q + q/2/mDM

# Energy spectrum per mass:
def dRdEe_halo(Ee, sigma_e, mDM, target):
    if target!="hgte" and target!="cdte":
        raise ValueError("target not recognized for JWST")
    elif target == "hgte":
        integral  = 0.0
        prefactor = rho_DM / mDM / M_cell * nu.aEM * sigma_e * nu.mElectron**2 / nu.Reduced_Mass(nu.mElectron, mDM)**2
        for qi in q_grid:
            vMin = v_min(qi, Ee, mDM)
            ff_qiE = ff_hgte((qi, Ee))
            integral += prefactor * dq / qi**2 * EtaFunction(vMin) * F_DM(qi)**2 * ff_qiE
    elif target == "cdte":
        integral  = 0.0
        prefactor = rho_DM / mDM / M_cell * nu.aEM * sigma_e * nu.mElectron**2 / nu.Reduced_Mass(nu.mElectron, mDM)**2
        for qi in q_grid:
            vMin = v_min(qi, Ee, mDM)
            ff_qiE = ff_cdte((qi, Ee))
            integral += prefactor * dq / qi**2 * EtaFunction(vMin) * F_DM(qi)**2 * ff_qiE
    return integral

# Charge Yield:
def charge_yield(Ee, Q):
    if Ee < energy_gap:
        return 0
    else:
        Ee_1 = epsilon * (Q - 1) + energy_gap
        Ee_2 = epsilon * Q + energy_gap
        if Ee < Ee_1 or Ee > Ee_2:
            return 0.0
        else:
            return 1.0

# Electron spectrum per mass:
def R_Q_halo(Q, sigma_e, mDM, target):
    R_Q = 0
    for Ei in E_grid:
        cy = charge_yield(Ei, Q)
        R_Q += dE * cy * dRdEe_halo(Ei, sigma_e, mDM, target)
    return R_Q

In [19]:
# JWST parameters
pixel_mass = 1.2e-8 *nu.gram
exposure_time   = 3574.278 *nu.sec * 244 / 245  # since we are using (last frame - first frame)
exposure = pixel_mass * exposure_time

log_m_min  = -3
log_m_max  = 1
n_m    = 33   # From 1e-3  to 10 GeV
m_grid      = np.logspace(log_m_min, log_m_max, n_m) * nu.GeV

#center line cross sections for light mediator

center_line = np.array([2.15504637e-23, 1.88334907e-23, 1.53030461e-23, 1.32738030e-23,
       1.26359147e-23, 1.37889395e-23, 1.61669130e-23, 1.95514385e-23,
       2.40008514e-23, 3.14137096e-23, 4.03532201e-23, 5.19960700e-23,
       6.95747264e-23, 9.69011074e-23, 1.40957345e-22, 2.05768490e-22,
       2.84518575e-22, 3.82210762e-22, 5.17381239e-22, 7.29305911e-22,
       1.08351297e-21, 1.55439713e-21, 2.15203017e-21, 3.00211451e-21,
       4.12016417e-21, 5.67974728e-21, 7.78952222e-21, 1.06757849e-20,
       1.45615810e-20, 1.99263396e-20, 2.70914228e-20, 3.67600191e-20,
       4.94000000e-20]) * nu.cm * nu.cm



In [20]:
def compute(j):
    m = m_grid[j]
    cs = center_line[j]
    nq = np.zeros(30)

    dRdEe_hgte = np.zeros(N_E)
    dRdEe_cdte = np.zeros(N_E)
    i = 0
    for Ei in E_grid:
        dRdEe_hgte[i] = dRdEe_halo(Ei, cs, m, "hgte")
        dRdEe_cdte[i] = dRdEe_halo(Ei, cs, m, "cdte")
        i += 1

    for q in range(30):
        Q = q+1
        print('calculating: i=' + str(j) + ' Q=' + str(Q) +'\n')

        nq_hgte = 0
        nq_cdte = 0
        i = 0
        for Ei in E_grid:
            cy = charge_yield(Ei, Q)
            nq_hgte +=  dE * cy * dRdEe_hgte[i]
            nq_cdte +=  dE * cy * dRdEe_cdte[i]
            i += 1

        nq_hgte *= exposure
        nq_cdte *= exposure
        nq[q] = np.min((nq_hgte, nq_cdte))
    
    np.savetxt('../data/binned_signal_halo_w_lindhard_finer/binned_signals_Halo_'+ str(j) + '.txt', nq, header=str(m))
    return 0

In [21]:
arr = np.linspace(0, 32, 33, dtype=int)
with concurrent.futures.ProcessPoolExecutor() as executor:
    executor.map(compute, arr)

calculating: i=14 Q=1

calculating: i=14 Q=2

calculating: i=14 Q=3

calculating: i=14 Q=4

calculating: i=14 Q=5

calculating: i=14 Q=6

calculating: i=14 Q=7

calculating: i=14 Q=8

calculating: i=14 Q=9

calculating: i=14 Q=10

calculating: i=14 Q=11

calculating: i=14 Q=12

calculating: i=14 Q=13

calculating: i=14 Q=14

calculating: i=14 Q=15

calculating: i=14 Q=16

calculating: i=14 Q=17

calculating: i=14 Q=18

calculating: i=14 Q=19

calculating: i=14 Q=20

calculating: i=14 Q=21

calculating: i=14 Q=22

calculating: i=14 Q=23

calculating: i=14 Q=24

calculating: i=14 Q=25

calculating: i=14 Q=26

calculating: i=14 Q=27

calculating: i=14 Q=28

calculating: i=14 Q=29

calculating: i=14 Q=30

calculating: i=4 Q=1

calculating: i=4 Q=2

calculating: i=4 Q=3

calculating: i=4 Q=4

calculating: i=4 Q=5

calculating: i=4 Q=6

calculating: i=4 Q=7

calculating: i=4 Q=8

calculating: i=4 Q=9

calculating: i=4 Q=10

calculating: i=4 Q=11

calculating: i=4 Q=12

calculating: i=4 Q=13
